# NLP

NLP, ou Processamento de Linguagem Natural, e uma area da ciencia de dados que estuda como transformar textos em informacao estruturada. Com essas tecnicas, conseguimos analisar noticias, identificar temas, contar palavras, comparar documentos e preparar dados textuais para modelos de aprendizado de maquina.

In [24]:
import pandas as pd

df = pd.read_csv("../Webscraping/data/noticias_palmeiras (1).csv", sep = ";")
df.head()

df['texto'] = df['texto'].astype(str)

df.head(5)


,titulo,data,texto,visitas
0,STJD indefere pedido de suspensão e Abel Ferre...,11/4/2026 16:47,O Palmeiras terá uma nova rodada de desafios n...,141
1,Nubank Anuncia Amistoso Imperdível: Palmeiras ...,11/4/2026 16:47,O Nubank está explorando a possibilidade de or...,129
2,STJD Recusa Suspensão de Abel Ferreira; Palmei...,11/4/2026 16:47,A Sociedade Esportiva Palmeiras manifestou sua...,105
3,Palmeiras encerra treinos para o Dérbi com Vit...,11/4/2026 14:27,"Na manhã deste sábado (11), a Academia de Fute...",276
4,Messi Pode Anunciar Amistoso Histórico em São ...,11/4/2026 14:07,"A co-fundadora do Nubank, Cristina Junqueira, ...",246


## Trabalhando apenas com o texto

Por enquanto, vamos trabalhar somente com a coluna `texto`, que contem o corpo completo de cada noticia. As outras colunas, como `titulo`, `descricao`, `tags` e `data`, continuam no `DataFrame`, mas vamos deixar elas de lado neste primeiro momento para entender melhor o conteudo textual.

## Passos da analise

Vamos preparar os textos aos poucos:

1. Limpar os textos.
2. Remover palavras muito comuns.
3. Criar uma representacao Bag of Words.
4. Contar palavras frequentes.
5. Transformar textos em numeros para analises posteriores.

## 1. Limpeza basica

Na celula abaixo:

- `wordpunct_tokenize` separa o texto em palavras e pontuacao.
- `texto.lower()` coloca tudo em minusculas.
- `unidecode(texto)` troca letras acentuadas por letras sem acento.
- `token.isalnum()` mantem apenas letras e numeros.
- `" ".join(tokens)` junta os tokens em uma frase limpa.
- `.apply(limpar_texto)` aplica a funcao em todas as noticias.

Exemplo: `"Olá, Senado!"` vira `"ola senado"`.

In [25]:
from nltk.tokenize import wordpunct_tokenize
from unidecode import unidecode


def limpar_texto(texto):
    texto = texto.lower()
    texto = unidecode(texto)
    tokens = wordpunct_tokenize(texto)
    tokens = [token for token in tokens if token.isalnum()]
    return " ".join(tokens)


df["texto_limpo"] = df["texto"].apply(limpar_texto)

df[["texto", "texto_limpo"]].head()

,texto,texto_limpo
0,O Palmeiras terá uma nova rodada de desafios n...,o palmeiras tera uma nova rodada de desafios n...
1,O Nubank está explorando a possibilidade de or...,o nubank esta explorando a possibilidade de or...
2,A Sociedade Esportiva Palmeiras manifestou sua...,a sociedade esportiva palmeiras manifestou sua...
3,"Na manhã deste sábado (11), a Academia de Fute...",na manha deste sabado 11 a academia de futebol...
4,"A co-fundadora do Nubank, Cristina Junqueira, ...",a co fundadora do nubank cristina junqueira re...


## 2. Removendo stopwords

Stopwords sao palavras muito comuns, como `a`, `o`, `de`, `para` e `que`. Elas aparecem muito, mas geralmente ajudam pouco a entender o tema de um texto.

Na celula abaixo:

- `stopwords.words("portuguese")` carrega stopwords em portugues.
- `texto.split()` separa o texto limpo em palavras.
- `token not in stopwords_pt` remove as palavras muito comuns.
- `.str.join(" ")` junta os tokens restantes em um texto sem stopwords.
- `.apply(remover_stopwords)` aplica a funcao em todas as noticias.

Exemplo: `"o senador falou com a imprensa"` vira `["senador", "falou", "imprensa"]`.

In [26]:
import nltk

from nltk.corpus import stopwords


nltk.download("stopwords", quiet=True)

stopwords_pt = stopwords.words("portuguese")
stopwords_pt = [unidecode(palavra) for palavra in stopwords_pt]
stopwords_pt = set(stopwords_pt)


def remover_stopwords(texto):
    tokens = texto.split()
    tokens = [token for token in tokens if token not in stopwords_pt]
    return tokens


df["tokens_sem_stopwords"] = df["texto_limpo"].apply(remover_stopwords)
df["texto_sem_stopwords"] = df["tokens_sem_stopwords"].str.join(" ")

df[["texto_limpo", "tokens_sem_stopwords", "texto_sem_stopwords"]].head()

,texto_limpo,tokens_sem_stopwords,texto_sem_stopwords
0,o palmeiras tera uma nova rodada de desafios n...,"[palmeiras, nova, rodada, desafios, campeonato...",palmeiras nova rodada desafios campeonato bras...
1,o nubank esta explorando a possibilidade de or...,"[nubank, explorando, possibilidade, organizar,...",nubank explorando possibilidade organizar amis...
2,a sociedade esportiva palmeiras manifestou sua...,"[sociedade, esportiva, palmeiras, manifestou, ...",sociedade esportiva palmeiras manifestou insat...
3,na manha deste sabado 11 a academia de futebol...,"[manha, deste, sabado, 11, academia, futebol, ...",manha deste sabado 11 academia futebol palco u...
4,a co fundadora do nubank cristina junqueira re...,"[co, fundadora, nubank, cristina, junqueira, r...",co fundadora nubank cristina junqueira revelou...


## 3. Bag of Words

No Bag of Words, cada linha representa uma noticia e cada coluna representa uma palavra. O valor indica quantas vezes aquela palavra apareceu na noticia.

Exemplo: `"senador falou senador"` teria `senador = 2` e `falou = 1`.

In [27]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()
matriz_bow = vectorizer.fit_transform(df["texto_sem_stopwords"])

df_bow = pd.DataFrame(
    matriz_bow.toarray(),
    columns=vectorizer.get_feature_names_out()
)

df_bow.head()

,00,000,00central,00para,00r,00superior,01,012,02,03,...,zerar,zero,zhu,zie,zona,zonas,zortea,zorzi,zubeldia,zvlr9u0adq
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [20]:
df_bow[df_bow["zvlr9u0adq"]>0]
#df_bow.columns

,00,000,00central,00para,00r,00superior,01,012,02,03,...,zerar,zero,zhu,zie,zona,zonas,zortea,zorzi,zubeldia,zvlr9u0adq
637,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


### Removendo colunas com numeros

Agora vamos remover qualquer coluna cujo nome tenha pelo menos um numero.

In [28]:
colunas_com_numeros = [col for col in df_bow.columns if any(char.isdigit() for char in col)]

df_bow = df_bow.drop(columns=colunas_com_numeros)

freq=df_bow.sum(axis=0)
colunas_validas = freq[freq>2].index
df_bow = df_bow[colunas_validas]

print(f"{len(colunas_com_numeros)} colunas removidas")
print(f"{len(freq) - len(colunas_validas)} colunas removidas")
df_bow.head()

505 colunas removidas
7895 colunas removidas


,abafa,abaixo,abalou,abandonar,abdicar,abel,aberta,abertamente,abertas,aberto,...,zaneratto,zanni,zanotti,ze,zenit,zerar,zero,zona,zorzi,zubeldia
0,0,0,0,0,0,3,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,3,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,3,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Criando o DataFrame final

Agora vamos juntar os metadados das noticias com as colunas do Bag of Words. Para evitar conflito de nomes, as colunas do Bag of Words recebem o prefixo `bow_`.

In [29]:
metadados = df[["data", "titulo", "visitas"]].reset_index(drop=True)
bow_com_prefixo = df_bow.add_prefix("bow_").reset_index(drop=True)

df_final = pd.concat([metadados, bow_com_prefixo], axis=1)

df_final.head()

,data,titulo,visitas,bow_abafa,bow_abaixo,bow_abalou,bow_abandonar,bow_abdicar,bow_abel,bow_aberta,...,bow_zaneratto,bow_zanni,bow_zanotti,bow_ze,bow_zenit,bow_zerar,bow_zero,bow_zona,bow_zorzi,bow_zubeldia
0,11/4/2026 16:47,STJD indefere pedido de suspensão e Abel Ferre...,141,0,0,0,0,0,3,0,...,0,0,0,0,0,0,0,0,0,0
1,11/4/2026 16:47,Nubank Anuncia Amistoso Imperdível: Palmeiras ...,129,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,11/4/2026 16:47,STJD Recusa Suspensão de Abel Ferreira; Palmei...,105,0,0,0,0,0,3,0,...,0,0,0,0,0,0,0,0,0,0
3,11/4/2026 14:27,Palmeiras encerra treinos para o Dérbi com Vit...,276,0,0,0,0,0,3,0,...,0,0,0,0,0,0,0,0,0,0
4,11/4/2026 14:07,Messi Pode Anunciar Amistoso Histórico em São ...,246,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Exemplo de analise

Agora podemos fazer uma analise simples das palavras do Bag of Words: quais aparecem mais, quais aparecem menos e quantas palavras diferentes temos no vocabulario.

In [23]:
colunas_bow = [coluna for coluna in df_final.columns if coluna.startswith("bow_")]

frequencia_palavras = df_final[colunas_bow].sum().sort_values(ascending=False)

print(f"Total de palavras diferentes: {len(frequencia_palavras)}")

frequencia_palavras.head(10)

Total de palavras diferentes: 8441


bow_palmeiras     8159
bow_equipe        3782
bow_jogo          2869
bow_campeonato    2742
bow_abel          2598
bow_clube         2353
bow_elenco        2283
bow_ferreira      2203
bow_apos          1964
bow_desempenho    1947
dtype: int64

In [24]:
frequencia_palavras.tail(10)

bow_depositar        3
bow_resistir         3
bow_autoconfianca    3
bow_autoestima       3
bow_deposita         3
bow_reservado        3
bow_resenha          3
bow_dependerao       3
bow_requisito        3
bow_estipulou        3
dtype: int64

In [25]:
documentos_por_palavra = (df_final[colunas_bow] > 0).sum().sort_values(ascending=False)
documentos_por_palavra.index = documentos_por_palavra.index.str.replace("bow_", "", regex=False)

documentos_por_palavra.head(10)

palmeiras     1960
elenco        1449
equipe        1446
jogo          1429
campeonato    1373
gestao        1334
desempenho    1256
apos          1249
abel          1201
ferreira      1199
dtype: int64

In [26]:
df_final["palavras_unicas"] = (df_final[colunas_bow] > 0).sum(axis=1)

df_final[["titulo", "palavras_unicas"]].sort_values("palavras_unicas", ascending=True).head(10)

,titulo,palavras_unicas
1823,TABELA ATUALIZADA! Verdão sobe na tabela após ...,0
967,Novorizontino x Palmeiras: melhores momentos,1
1302,TABELA ATUALIZADA! Veja a tabela após grande v...,1
800,TABELA ATUALIZADA! Veja a tabela após derrota ...,1
1533,Palmeiras x Guarani: melhores momentos,1
1723,OLHO NA TABELA! Palmeiras se aproxima da lider...,1
630,TABELA ATUALIZADA! Veja a tabela após grande v...,1
640,ESCALAÇÃO! Palmeiras pronto e escalado para en...,1
1407,Palmeiras x Capivariano: melhores momentos,1
1610,SEGUE O LÍDER! Tabela atualizada e Verdão volt...,1


## TDI E TDF

In [30]:
from sklearn.feature_extraction.text import TfidfTransformer
import pandas as pd

# transforma BOW -> TF-IDF
transformer = TfidfTransformer()

matriz_tfidf = transformer.fit_transform(df_bow)

# converte para DataFrame
df_tfidf = pd.DataFrame(
    matriz_tfidf.toarray(),
    columns=df_bow.columns
)

print(df_tfidf.head())

   abafa  abaixo  abalou  abandonar  abdicar      abel  aberta  abertamente  \
0    0.0     0.0     0.0        0.0      0.0  0.075833     0.0          0.0   
1    0.0     0.0     0.0        0.0      0.0  0.000000     0.0          0.0   
2    0.0     0.0     0.0        0.0      0.0  0.070553     0.0          0.0   
3    0.0     0.0     0.0        0.0      0.0  0.068345     0.0          0.0   
4    0.0     0.0     0.0        0.0      0.0  0.000000     0.0          0.0   

   abertas  aberto  ...  zaneratto  zanni  zanotti   ze  zenit  zerar  zero  \
0      0.0     0.0  ...        0.0    0.0      0.0  0.0    0.0    0.0   0.0   
1      0.0     0.0  ...        0.0    0.0      0.0  0.0    0.0    0.0   0.0   
2      0.0     0.0  ...        0.0    0.0      0.0  0.0    0.0    0.0   0.0   
3      0.0     0.0  ...        0.0    0.0      0.0  0.0    0.0    0.0   0.0   
4      0.0     0.0  ...        0.0    0.0      0.0  0.0    0.0    0.0   0.0   

   zona  zorzi  zubeldia  
0   0.0    0.0       0.

In [31]:
df_tfidf.sum(axis=0).sort_values()

cobram               0.055380
pokerlistings        0.055380
praticidade          0.055380
compatibilidade      0.055380
depositar            0.055380
                      ...    
abel                66.636745
jogo                67.067720
campeonato          69.310295
equipe              91.496395
palmeiras          145.268954
Length: 8441, dtype: float64

In [32]:
maiores_palavras = df_tfidf.idxmax(axis=1)

maior_valor = df_tfidf.max(axis=1)

resultado = pd.DataFrame({
    "palavra": maiores_palavras,
    "tfidf": maior_valor
})

print(resultado.head(50))

           palavra     tfidf
0         tribunal  0.233586
1            messi  0.279538
2         entidade  0.170296
3          recorde  0.200418
4           nubank  0.349891
5            messi  0.340579
6             sosa  0.433455
7        barcellos  0.578157
8            derbi  0.217531
9           nubank  0.431597
10       classicos  0.167242
11          nubank  0.304309
12          visual  0.329927
13           derbi  0.166148
14        garrides  0.572455
15          kansas  0.561833
16         destaca  0.211076
17           jesus  0.570787
18        classico  0.212813
19            stjd  0.243841
20          nubank  0.514395
21          nubank  0.597064
22           jesus  0.531484
23       suspensao  0.268063
24          nubank  0.513863
25          nubank  0.262992
26          nubank  0.318654
27          nubank  0.452390
28          nubank  0.454865
29          artigo  0.229455
30      suspensivo  0.199867
31          nubank  0.302118
32         allianz  0.297094
33          nu

In [33]:
from pathlib import Path

pasta_dados = Path("../dados")
pasta_dados.mkdir(exist_ok=True)

# Dados completos: metadados + texto original + texto limpo + texto sem stopwords
df_completo = df[[ "titulo", "data", "visitas", "texto", "texto_limpo", "texto_sem_stopwords"]]
df_completo.to_csv(pasta_dados / "noticias.csv", index=False)

# Bag of Words (metadados + colunas bow_)
df_final.to_csv(pasta_dados / "bow.csv", index=False)

# TF-IDF (metadados + colunas tfidf_)
df_tfidf.to_csv(pasta_dados / "tfidf.csv", index=False)

print(f"Noticias completas: {pasta_dados / 'noticias.csv'} (shape={df_completo.shape})")
print(f"Bag of Words:       {pasta_dados / 'bow.csv'}      (shape={df_final.shape})")
print(f"TF-IDF:             {pasta_dados / 'tfidf.csv'}    (shape={df_tfidf.shape})")

Noticias completas: ../dados/noticias.csv (shape=(2000, 6))
Bag of Words:       ../dados/bow.csv      (shape=(2000, 8444))
TF-IDF:             ../dados/tfidf.csv    (shape=(2000, 8441))
